# Churn

Compare logo churn and revenue churn from the Tidemill engine with
ground-truth values computed directly from Stripe subscription lifecycle data.

## How churn is measured

**Logo (customer) churn** — the fraction of *customers* who fully cancel:

```
logo_churn = customers_fully_churned / customers_active_at_period_start
```

A customer is "active at start" if they had at least one subscription created
before the period start that was not yet canceled.  They are "fully churned"
if, by the period end, they have zero active subscriptions remaining.

**Revenue churn** — the fraction of *MRR* lost to cancellations:

```
revenue_churn = |churned_MRR| / starting_MRR
```

`churned_MRR` sums the `mrr_cents` of every subscription whose `canceled_at`
timestamp falls within `[start, end)`.

## Stripe fields used

- `subscription.status` — "canceled" means the sub ended
- `subscription.canceled_at` — Unix timestamp of cancellation
- `subscription.created` — Unix timestamp of creation
- `subscription.items.data[*].price` — used to compute per-sub MRR

The denominator matters: both Tidemill and this notebook use the same
starting MRR (from Tidemill's waterfall) so the comparison is apples-to-apples.

In [1]:
import os

import stripe

from tidemill import reports
from tidemill.reports.stripecheck import StripeData, TidemillClient

stripe.api_key = os.environ["STRIPE_API_KEY"]
reports.setup()

START, END = "2025-09-01", "2026-03-31"
CHURN_START = "2025-10-01"  # need a prior month as baseline denominator
tm = TidemillClient()
sd = StripeData()

## 1. Churn comparison — Tidemill vs Stripe

`stripecheck.compare.churn()` queries both sources over the same window
and uses the same starting-MRR denominator for revenue churn.

In [2]:
data = reports.churn.stripe_overview(tm, sd, CHURN_START, END)
reports.churn.style_stripe_overview(data)

Metric,Tidemill,Stripe,Match
Logo churn,22.2%,22.2%,True
Revenue churn,21.1%,28.7%,False
Active at start,1800.0%,1800.0%,True
Fully churned,400.0%,400.0%,True


In [3]:
reports.churn.plot_stripe_overview(data)

## 2. Churn from Stripe (detailed)

`stripecheck.stripe_metrics.churn_rates()` implements the per-customer
analysis:

1. Group subscriptions by `customer` ID
2. For each customer, check if they had active subs at period start
3. Check if they still have active subs at period end
4. Sum MRR of subscriptions canceled within the window

The "active at start" check uses:
```python
created_at < start AND (canceled_at IS NULL OR canceled_at >= start)
```

In [4]:
from tidemill.reports.stripecheck.stripe_metrics import churn_rates

stripe_churn = churn_rates(sd.subscriptions, CHURN_START, END)
print("Stripe churn detail:")
for k, v in stripe_churn.items():
    if isinstance(v, float):
        print(f"  {k:25s}  {v * 100:.1f}%")
    else:
        print(f"  {k:25s}  {v}")

Stripe churn detail:
  logo_churn                 22.2%
  revenue_churn              15.8%
  active_at_start            18
  fully_churned              4
  churned_mrr_cents          14200
  starting_mrr_cents         89933


## 3. Churn visualization

In [5]:
reports.churn.plot_stripe_detail(stripe_churn)

## 4. Churn events from Tidemill vs Stripe cancellations

In [6]:
cancellations = reports.churn.stripe_cancellations(sd, CHURN_START, END)
reports.churn.style_stripe_cancellations(cancellations)

id,customer,canceled_month,mrr
sub_1TLTnNCMLOTbZAd7TkjREn9p,cus_UK83SrxmMLJWVN,2025-10,$21.00
sub_1TLTnJCMLOTbZAd7mycy2Ijs,cus_UK83tJS3WtBDVM,2025-10,$21.00
sub_1TLTmaCMLOTbZAd7uAxWFKqh,cus_UK82F2IfibEKGK,2025-10,$79.00
sub_1TLTmTCMLOTbZAd7fKGKwlWh,cus_UK82jvHurByzDc,2025-10,$21.00
sub_1TLToPCMLOTbZAd7hul4bT4I,cus_UK84QBWbSXjhdK,2025-11,$21.00
sub_1TLTpUCMLOTbZAd7xwQjIw6S,cus_UK85EHaGONijIZ,2025-12,$21.00
sub_1TLTpQCMLOTbZAd7DgG57Fap,cus_UK85qxh4rssA05,2025-12,$21.00
sub_1TLTqwCMLOTbZAd74BFNmZEa,cus_UK87sGG5CU458q,2026-01,$21.00
sub_1TLTsYCMLOTbZAd7FOevjiit,cus_UK88lxsfZAzw9w,2026-02,$21.00
sub_1TLTsUCMLOTbZAd7WZ0raNZK,cus_UK88hoeBxCDNYO,2026-02,$21.00


## 5. Monthly Churn Rate Timeline

Compute logo and revenue churn rates per month to visualize churn trends over time.

In [7]:
churn_df = reports.churn.timeline(tm, CHURN_START, END)
reports.churn.style_timeline(churn_df)

,logo_churn,revenue_churn
month,,
2025-10,11.1%,6.2%
2025-11,13.6%,13.1%
2025-12,9.1%,5.3%
2026-01,3.8%,2.3%
2026-02,6.7%,4.2%


In [8]:
reports.churn.plot_timeline(churn_df)

## 6. Monthly Churned MRR

Lost MRR per month from the MRR waterfall — shows the dollar impact of churn over time.

In [9]:
reports.churn.plot_monthly_lost_mrr(reports.churn.monthly_lost_mrr(tm, START, END))